# 😴 Prediksi Gangguan Tidur Berdasarkan Gaya Hidup
### Pendekatan CRISP-DM
**Dataset:** Sleep Health and Lifestyle (374 data) | **Target:** None / Insomnia / Sleep Apnea | **Model:** Random Forest

---
## 📦 Import Library

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import classification_report, accuracy_score, ConfusionMatrixDisplay
import warnings
warnings.filterwarnings('ignore')
print('✅ Library berhasil diimport')

---
## 1️⃣ Business Understanding

### Latar Belakang
Gangguan tidur seperti insomnia dan sleep apnea semakin umum di masyarakat modern dan berkaitan erat dengan gaya hidup seseorang. Deteksi dini dapat membantu seseorang segera mencari penanganan yang tepat.

### Problem Statement
> *Dapatkah kita memprediksi jenis gangguan tidur seseorang (None, Insomnia, Sleep Apnea) berdasarkan data gaya hidup dan kondisi kesehatan?*

### Goals
- Membangun model klasifikasi untuk memprediksi jenis gangguan tidur
- Mengidentifikasi faktor gaya hidup yang paling berpengaruh
- Men-deploy model sebagai aplikasi web interaktif

### Solution Statement
- **Model Utama:** Random Forest Classifier
- **Model Pembanding:** Gradient Boosting Classifier
- **Metrik:** Accuracy, Precision, Recall, F1-Score

---
## 2️⃣ Data Understanding

In [ ]:
df = pd.read_csv('Sleep_health_and_lifestyle_dataset.csv')
print(f'Shape: {df.shape}')
df.head(10)

In [ ]:
df.info()

In [ ]:
df.describe()

In [ ]:
print('Missing Values:')
print(df.isnull().sum())
print('\nCatatan: NaN di Sleep Disorder = tidak punya gangguan tidur (None)')

In [ ]:
print(f'Duplikat: {df.duplicated().sum()}')

In [ ]:
# Isi NaN dengan None untuk visualisasi
df['Sleep Disorder'] = df['Sleep Disorder'].fillna('None')

plt.figure(figsize=(7,4))
df['Sleep Disorder'].value_counts().plot(kind='bar', color=['#2ecc71','#e74c3c','#3498db'], edgecolor='white')
plt.title('Distribusi Sleep Disorder', fontsize=14)
plt.xlabel('Jenis Gangguan Tidur')
plt.ylabel('Jumlah')
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()
print(df['Sleep Disorder'].value_counts())

In [ ]:
fig, axes = plt.subplots(1,3, figsize=(15,4))
df['Gender'].value_counts().plot(kind='bar', ax=axes[0], color=['#3498db','#e74c3c'], edgecolor='white')
axes[0].set_title('Distribusi Gender'); axes[0].tick_params(axis='x', rotation=0)
df['BMI Category'].value_counts().plot(kind='bar', ax=axes[1], color='#9b59b6', edgecolor='white')
axes[1].set_title('Distribusi BMI'); axes[1].tick_params(axis='x', rotation=15)
df['Occupation'].value_counts().plot(kind='bar', ax=axes[2], color='#e67e22', edgecolor='white')
axes[2].set_title('Distribusi Pekerjaan'); axes[2].tick_params(axis='x', rotation=90)
plt.tight_layout(); plt.show()

In [ ]:
numerical_cols = ['Age','Sleep Duration','Quality of Sleep','Physical Activity Level','Stress Level','Heart Rate','Daily Steps']
plt.figure(figsize=(10,7))
sns.heatmap(df[numerical_cols].corr(), annot=True, fmt='.2f', cmap='coolwarm', linewidths=0.5)
plt.title('Heatmap Korelasi Fitur Numerik', fontsize=14)
plt.tight_layout(); plt.show()

In [ ]:
plt.figure(figsize=(7,4))
sns.boxplot(data=df, x='Sleep Disorder', y='Stress Level', palette={'None':'#2ecc71','Insomnia':'#e74c3c','Sleep Apnea':'#3498db'})
plt.title('Stress Level per Jenis Gangguan Tidur', fontsize=13)
plt.tight_layout(); plt.show()

---
## 3️⃣ Data Preparation

In [ ]:
df_clean = df.copy()
df_clean = df_clean.drop(columns=['Person ID'])
print(f'Kolom: {df_clean.columns.tolist()}')

In [ ]:
# Pecah Blood Pressure jadi dua kolom
df_clean[['BP Systolic','BP Diastolic']] = df_clean['Blood Pressure'].str.split('/', expand=True).astype(int)
df_clean = df_clean.drop(columns=['Blood Pressure'])
print('✅ Blood Pressure dipecah jadi BP Systolic & BP Diastolic')
df_clean[['BP Systolic','BP Diastolic']].head()

In [ ]:
# Encode kolom kategorikal
le = LabelEncoder()
categorical_cols = ['Gender','Occupation','BMI Category']
encoding_maps = {}
for col in categorical_cols:
    df_clean[col] = le.fit_transform(df_clean[col])
    encoding_maps[col] = dict(zip(le.classes_, le.transform(le.classes_)))
    print(f'{col}: {encoding_maps[col]}')

# Simpan encoding map untuk dipakai di app.py
print('\n✅ Encoding selesai')

In [ ]:
feature_cols = [
    'Gender','Age','Occupation','Sleep Duration',
    'Quality of Sleep','Physical Activity Level','Stress Level',
    'BMI Category','Heart Rate','Daily Steps',
    'BP Systolic','BP Diastolic'
]
X = df_clean[feature_cols]
y = df_clean['Sleep Disorder']
print(f'Shape X: {X.shape}, Shape y: {y.shape}')
print(f'Kelas: {y.unique()}')

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
print(f'Train: {X_train.shape[0]} | Test: {X_test.shape[0]}')

---
## 4️⃣ Modeling

In [ ]:
rf_model = RandomForestClassifier(n_estimators=100, max_depth=10, random_state=42, n_jobs=-1)
rf_model.fit(X_train, y_train)
print('✅ Random Forest selesai')

In [ ]:
gb_model = GradientBoostingClassifier(n_estimators=100, max_depth=5, random_state=42)
gb_model.fit(X_train, y_train)
print('✅ Gradient Boosting selesai')

---
## 5️⃣ Evaluation

In [ ]:
rf_pred = rf_model.predict(X_test)
gb_pred = gb_model.predict(X_test)
print(f'Accuracy Random Forest    : {accuracy_score(y_test, rf_pred):.4f}')
print(f'Accuracy Gradient Boosting: {accuracy_score(y_test, gb_pred):.4f}')

In [ ]:
print('=== Random Forest ===')
print(classification_report(y_test, rf_pred))
print('=== Gradient Boosting ===')
print(classification_report(y_test, gb_pred))

In [ ]:
fig, axes = plt.subplots(1,2, figsize=(14,5))
ConfusionMatrixDisplay.from_predictions(y_test, rf_pred, cmap='Greens', ax=axes[0])
axes[0].set_title('Confusion Matrix: Random Forest')
ConfusionMatrixDisplay.from_predictions(y_test, gb_pred, cmap='Blues', ax=axes[1])
axes[1].set_title('Confusion Matrix: Gradient Boosting')
plt.tight_layout(); plt.show()

In [ ]:
importances = pd.Series(rf_model.feature_importances_, index=feature_cols).sort_values(ascending=True)
plt.figure(figsize=(8,6))
importances.plot(kind='barh', color='#2ecc71', edgecolor='white')
plt.title('Feature Importance - Random Forest', fontsize=14)
plt.xlabel('Importance Score')
plt.tight_layout(); plt.show()

In [ ]:
cv_scores = cross_val_score(rf_model, X, y, cv=5, scoring='accuracy', n_jobs=-1)
print(f'CV Scores: {cv_scores}')
print(f'Mean Accuracy: {cv_scores.mean():.4f} ± {cv_scores.std():.4f}')

---
## 6️⃣ Deployment

In [ ]:
import joblib
joblib.dump(rf_model, 'sleep_model.pkl')
print('✅ Model disimpan sebagai sleep_model.pkl')

In [ ]:
# Test prediksi
loaded_model = joblib.load('sleep_model.pkl')
sample = pd.DataFrame([{
    'Gender': 1, 'Age': 30, 'Occupation': 8,
    'Sleep Duration': 6.0, 'Quality of Sleep': 5,
    'Physical Activity Level': 40, 'Stress Level': 7,
    'BMI Category': 2, 'Heart Rate': 80,
    'Daily Steps': 5000, 'BP Systolic': 130, 'BP Diastolic': 85
}])
print(f'Contoh Prediksi: {loaded_model.predict(sample)[0]}')

---
⚠️ **Penting:** Catat hasil encoding dari cell sebelumnya (Gender, Occupation, BMI Category) karena dibutuhkan untuk app.py